# Unit 3 — Space Invaders: Deep Q-Learning (DQN)

**Pass criterion:** `mean_reward − std_reward ≥ 200`

The Q-table from Unit 2 is replaced by a **CNN**: it takes the screen as input and predicts, for every button, "how many points will I collect if I press this now".

**Notes**
- **Use a GPU:** `Runtime → Change runtime type → T4 GPU`.
- Training takes **~2–3 hours**. The model is **checkpointed to Google Drive every 100k steps**; if Colab disconnects, `Run all` again and training resumes from the latest checkpoint.
- Google Drive access is requested once at the start.

**How to run:** enable the GPU → `Runtime → Run all` → allow Drive access. The Hugging Face login reads the `HF_TOKEN` Colab secret.

## 1. Setup

In [ ]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" ale-py huggingface_sb3 imageio imageio-ffmpeg moviepy

## 2. Settings and Google Drive

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "SpaceInvadersNoFrameskip-v4"
MODEL_NAME = f"dqn-{ENV_ID}"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"
TOTAL_STEPS = 2_000_000          # total environment steps

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/deep-rl-course/unit3"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import glob, re, imageio, torch
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CheckpointCallback

gym.register_envs(ale_py)
print("GPU:", torch.cuda.is_available())

import base64
from IPython.display import HTML, display

def show_video(path, width=500):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 3. The environment
The raw frame is 210×160 RGB. `make_atari_env` applies the standard DQN **preprocessing**:
- **Grayscale + resize to 84×84:** colour is not needed here; smaller images mean faster learning.
- **Frame skip = 4:** the agent makes a decision every 4 frames.
- **Reward clipping:** during training every reward becomes −1, 0 or +1 so that high-scoring games don't destabilise learning. (Evaluation uses the real game score.)

`VecFrameStack(n_stack=4)`: **the last 4 frames are stacked** so the network can perceive motion.

In [ ]:
def make_env(seed=0, render=False):
    kwargs = {"render_mode": "rgb_array"} if render else {}
    env = make_atari_env(ENV_ID, n_envs=1, seed=seed, env_kwargs=kwargs)
    return VecFrameStack(env, n_stack=4)

env = make_env()
print("Observation:", env.observation_space.shape, "(4 stacked 84x84 frames)")
print("Actions    :", env.action_space.n, "->", gym.make(ENV_ID).unwrapped.get_action_meanings())

## 4. Video helper
Plays a full game (until all lives are lost) and records a video. `model=None` → random buttons.

In [ ]:
def play_and_record(model=None, seed=0, name="agent", max_frames=4000):
    ev = make_env(seed=seed, render=True)
    obs = ev.reset()
    frames, score = [ev.render()], None
    for _ in range(max_frames):
        if model is None:
            action = [ev.action_space.sample()]
        else:
            action, _ = model.predict(obs, deterministic=False)
        obs, _, _, infos = ev.step(action)
        frames.append(ev.render())
        if "episode" in infos[0]:          # full game over (all lives lost)
            score = infos[0]["episode"]["r"]
            break
    ev.close()
    path = f"{name}.mp4"
    imageio.mimsave(path, frames, fps=30, macro_block_size=1)
    print(f"Game score: {score}" if score is not None else f"(video stopped after {max_frames} steps, game still going)")
    show_video(path, width=320)

play_and_record(model=None, name="random_agent")

## 5. The DQN model
- `CnnPolicy`: input 4×84×84 → CNN → Q-values for the 6 buttons (the "neural network instead of a table" idea).
- `buffer_size=100_000`: the **experience replay** buffer; the last 100k transitions are stored and sampled at random.
- `learning_starts=100_000`: fill the buffer before learning starts.
- `target_update_interval=1000`: the **target network** is refreshed every 1000 steps (moving-target problem).
- `train_freq=4`: one gradient step every 4 environment steps.
- `exploration_fraction=0.1`, `exploration_final_eps=0.01`: ε decays from 1.0 to 0.01 over the first 10% of training (ε-greedy).
- `optimize_memory_usage=True`: stores the buffer with half the RAM so it fits in Colab.

**Resuming:** if a checkpoint exists on Drive it is loaded. The replay buffer is not saved, so the first 50k steps refill it.

In [ ]:
def latest_checkpoint():
    files = glob.glob(f"{SAVE_DIR}/dqn_*_steps.zip")
    if not files:
        return None
    return max(files, key=lambda f: int(re.search(r"_(\d+)_steps", f).group(1)))

ckpt = latest_checkpoint()
if ckpt:
    print("Resuming from", ckpt)
    model = DQN.load(ckpt, env=env, device="auto")
    model.learning_starts = model.num_timesteps + 50_000   # refill the empty replay buffer first
else:
    print("Starting from scratch")
    model = DQN(
        "CnnPolicy", env,
        buffer_size=100_000,
        learning_rate=1e-4,
        batch_size=32,
        learning_starts=100_000,
        target_update_interval=1000,
        train_freq=4,
        gradient_steps=1,
        exploration_fraction=0.1,
        exploration_final_eps=0.01,
        optimize_memory_usage=True,
        replay_buffer_kwargs=dict(handle_timeout_termination=False),
        verbose=0,
    )
print("Steps done so far:", model.num_timesteps)

## 6. Training (~2–3 hours)
Keep the tab open. If the runtime disconnects: `Runtime → Run all` → training resumes.

In [ ]:
remaining = TOTAL_STEPS - model.num_timesteps
checkpoint_cb = CheckpointCallback(save_freq=100_000, save_path=SAVE_DIR, name_prefix="dqn")
if remaining > 0:
    model.learn(total_timesteps=remaining, callback=checkpoint_cb,
                reset_num_timesteps=False, progress_bar=True)
model.save(f"{SAVE_DIR}/{MODEL_NAME}")
print("Training finished at", model.num_timesteps, "steps")

## 7. Evaluate
10 full games (3 lives each), **real game score**, no reward clipping.

In [ ]:
eval_env = make_env(seed=123, render=True)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=False)
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= 200 else 'NOT YET ❌ (increase TOTAL_STEPS by 500_000 and Run all again)'}")

## 8. Watch the trained agent

In [ ]:
play_and_record(model, seed=0, name="trained_agent")

## 9. Push to the Hugging Face Hub
The login cell reads the `HF_TOKEN` Colab secret (**Write** token).

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_sb3 import package_to_hub

package_to_hub(
    model=model,
    model_name=MODEL_NAME,
    model_architecture="DQN",
    env_id=ENV_ID,
    eval_env=make_env(seed=123, render=True),
    repo_id=REPO_ID,
    commit_message="Upload DQN SpaceInvaders agent",
)
print(f"Done: https://huggingface.co/{REPO_ID}")

## 10. Check
https://huggingface.co/spaces/ThomasSimonini/Check-my-progress-Deep-RL-Course → **Unit 3** ✅

Afterwards you can delete the `deep-rl-course/unit3` folder on Drive to free space.

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("SpaceInvadersNoFrameskip-v4", "stable-baselines3", 200)